# VMMR EfficientNet-B4 Testing Notebook
Upload any image from your device and run inference using your trained checkpoint.

In [ ]:
# Imports
import os
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
from google.colab import files

ImageFile.LOAD_TRUNCATED_IMAGES = True


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Paths (matching your training notebook)
CKPT_DIR = '/content/drive/MyDrive/vmmr_checkpoints'
BEST_CKPT_PATH = os.path.join(CKPT_DIR, 'efficientnet_b4_best.pth')
CLASSES_PATH = os.path.join(CKPT_DIR, 'classes.txt')

assert os.path.isfile(BEST_CKPT_PATH), f'Checkpoint not found: {BEST_CKPT_PATH}'
assert os.path.isfile(CLASSES_PATH), f'classes.txt not found: {CLASSES_PATH}'

print('Checkpoint:', BEST_CKPT_PATH)
print('Classes:', CLASSES_PATH)


In [ ]:
# Preprocessing config from training notebook
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMG_SIZE = 224

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

with open(CLASSES_PATH, 'r', encoding='utf-8') as f:
    class_names = [line.strip() for line in f if line.strip()]
num_classes = len(class_names)
print('Loaded classes:', num_classes)


In [ ]:
# Build model and load checkpoint
model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)

ckpt = torch.load(BEST_CKPT_PATH, map_location=device)
state_dict = ckpt.get('model_state', ckpt)
model.load_state_dict(state_dict)
model = model.to(device).eval()

ckpt_classes = ckpt.get('class_names', None) if isinstance(ckpt, dict) else None
if ckpt_classes is not None and list(ckpt_classes) != class_names:
    print('Warning: class_names in checkpoint differ from classes.txt. Using classes.txt order.')

print('Model loaded successfully.')


In [ ]:
# Inference helpers
def load_rgb_image(path: str):
    with Image.open(path) as img:
        return img.convert('RGB')

@torch.no_grad()
def predict_topk(pil_img, k=5):
    x = eval_tfms(pil_img).unsqueeze(0).to(device)
    logits = model(x)
    probs = torch.softmax(logits, dim=1).squeeze(0)

    k = min(k, num_classes)
    top_prob, top_idx = torch.topk(probs, k=k)
    top_prob = top_prob.detach().cpu().tolist()
    top_idx = top_idx.detach().cpu().tolist()
    return [(class_names[i], float(p)) for i, p in zip(top_idx, top_prob)]


In [ ]:
# Upload image(s) from your device and predict
# Supports any image format PIL can decode (jpg, jpeg, png, webp, bmp, tif, etc.)
uploaded = files.upload()

if not uploaded:
    print('No files uploaded.')
else:
    for fname in uploaded.keys():
        try:
            img = load_rgb_image(fname)
            preds = predict_topk(img, k=5)
            top1_class, top1_prob = preds[0]

            plt.figure(figsize=(7, 7))
            plt.imshow(img)
            plt.axis('off')
            plt.title(f'Input: {fname}')

            # Write top-1 prediction on image
            label_text = f"Pred: {top1_class} ({top1_prob*100:.2f}%)"
            plt.text(
                10,
                30,
                label_text,
                color='white',
                fontsize=12,
                bbox=dict(facecolor='black', alpha=0.7, pad=6)
            )
            plt.show()

            print(f'\nTop-5 predictions for: {fname}')
            for rank, (cls_name, prob) in enumerate(preds, 1):
                print(f'{rank}. {cls_name:<30}  {prob:.4f}')
        except Exception as e:
            print(f'Failed on {fname}: {e}')
